# Model_Test · One random article per run

Load the frozen Naive Bayes pipeline and external article sample. Each execution of the classification cell randomly selects **one** article, predicts its label, and displays the dataset's actual label beside the model prediction. Rerun that cell to draw another article.

This is an interactive example from the small, imbalanced, COVID-19-focused CrossFake sample. A single prediction does not measure overall model performance or establish that a story is factually true.


## 1. Load the model and external data

The pipeline expects the same input used during training: title and body joined with two newline characters. The actual label is kept aside for display after prediction and is never passed to the model.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import secrets

import joblib
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'raw' / 'News').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from data/raw/News.')

data_path = PROJECT_ROOT / 'data' / 'external' / 'Test_News' / 'external_test_news.csv'
model_path = PROJECT_ROOT / 'models' / 'final_naive_bayes_pipeline.joblib'
for path in [data_path, model_path]:
    if not path.exists():
        raise FileNotFoundError(f'Required data/model artifact is missing: {path}')

external = pd.read_csv(data_path, keep_default_na=False)
required_columns = {'article_id', 'title', 'text', 'label'}
missing_columns = required_columns - set(external.columns)
if missing_columns:
    raise ValueError(f'External data is missing columns: {sorted(missing_columns)}')
if external.empty or external['article_id'].duplicated().any():
    raise ValueError('External data must contain rows with unique article IDs.')
if not set(external['label']).issubset({'fake', 'real'}):
    raise ValueError('Labels must be standardized to `fake` and `real`.')
if external[['title', 'text']].eq('').all(axis=1).any():
    raise ValueError('An article has both an empty title and body.')

external['model_text'] = external['title'] + '\n\n' + external['text']
pipeline = joblib.load(model_path)
print(f'Loaded {len(external)} external articles and the saved pipeline.')
print('No training or tuning is performed in this notebook.')


Loaded 95 external articles and the saved pipeline.
No training or tuning is performed in this notebook.


## 2. Randomly select and classify one article

This cell chooses a fresh random row each time it runs. The table shows that single article's dataset label and predicted label. The CSV saved below is overwritten on each run, so it contains only the latest one-article result.


In [2]:
random_index = secrets.randbelow(len(external))
selected = external.iloc[random_index]
predicted_label = str(pipeline.predict([selected['model_text']])[0])
if predicted_label not in {'fake', 'real'}:
    raise ValueError(f'Unexpected prediction label: {predicted_label}')

actual_label = str(selected['label'])
one_article_result = pd.DataFrame([{
    'article_id': selected['article_id'],
    'title': selected['title'],
    'actual_label': actual_label,
    'predicted_label': predicted_label,
    'correct': actual_label == predicted_label,
}])
print(f'Randomly selected article {random_index + 1} of {len(external)}:')
display(one_article_result)


Randomly selected article 57 of 95:


,article_id,title,actual_label,predicted_label,correct
0,real_71,World experts and funders set priorities for C...,real,real,True


## 3. Save the latest single prediction

The result file contains one row for the most recently classified article. Run the classification cell again to randomly choose and classify another article; then rerun this save cell to replace the saved result.


In [3]:
output_dir = PROJECT_ROOT / 'outputs' / 'external_test'
output_dir.mkdir(parents=True, exist_ok=True)
result_path = output_dir / 'single_article_prediction.csv'
one_article_result = one_article_result.assign(
    evaluated_at_utc=datetime.now(timezone.utc).isoformat()
)
one_article_result.to_csv(result_path, index=False, encoding='utf-8')
print('Saved the latest one-article result:', result_path)


Saved the latest one-article result: C:\Projects_Tracking\News_Spam_Classifier\outputs\external_test\single_article_prediction.csv


### Interpretation

The displayed `actual_label` is the label supplied by the external dataset; `predicted_label` is the saved Naive Bayes model's output. These dataset labels are not a new independent fact-check. Use Notebook 4 and its full holdout results for aggregate model evaluation; one random row at a time is for inspecting individual classifications only.
